In [ ]:
# Street view
!git clone https://github.com/KI4Kids/bike.git

In [ ]:
pip install ultralytics

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/bike/

/content/bike


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
#results = model("helmet.mp4", save=True, show=True)
#results = model("ub_arch.mp4", save=True, show=True)
results = model("autos1.mp4", save=True, show=True)
#results = model("baustelle.mp4", save=True, show=True)

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Pfad zum Video
#video_path = '/content/bike/runs/detect/predict1/bike.avi'
#video_path = '/content/bike/runs/detect/predict2/bike.avi'
video_path = '/content/bike/runs/detect/predict/autos1.avi'

# AVI in MP4 umwandeln, da Browser AVI oft nicht unterstützt
!ffmpeg -i {video_path} -vcodec libx264 -f mp4 /content/output_video.mp4 -y

# MP4 kodieren und im HTML anzeigen
mp4_path = '/content/output_video.mp4'
mp4 = open(mp4_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

# HTML-Player anzeigen
HTML(f"""
<video width=640 controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")


In [ ]:
# Installiere notwendige Bibliotheken in Colab
!pip install ultralytics opencv-python

import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# Pfad zu deinem Video
video_path = 'https://frankyhub.de/KI_mp4/autos.mp4'  # Oder eine URL für einen Webcam-Stream

# Öffne das Video
cap = cv2.VideoCapture(video_path)

# Videoauflösung und FPS festlegen
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# Funktion zur Änderung der Videoauflösung
def change_res(width, height):
    cap.set(3, width)
    cap.set(4, height)

# It's generally better to keep the original resolution or resize proportionally
# change_res(1200, 720)

# Lade das YOLO-Modell von Ultralytics
model = YOLO("yolov5s.pt")  # Modell für schnelle Objekterkennung (z.B. YOLOv5s)

# Zähler
fcount = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)

    # Autos zählen (Klasse 2 im COCO-Dataset)
    ccount = 0
    if results[0].boxes is not None:
        classes = results[0].boxes.cls.cpu().numpy()  # Klassen-ID als NumPy-Array
        for cls in classes:
            if int(cls) == 2:
                ccount += 1
    fcount = ccount

    # Bounding Boxes rendern
    frame_with_boxes = results[0].plot()

    # Text anzeigen
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(frame_with_boxes, f" {fcount} Autos", (30, 50), font, 1, (0, 255, 0), 2, cv2.LINE_AA)

    # Frame anzeigen
    cv2_imshow(frame_with_boxes)
